In [6]:
import pandas as pd

combined = pd.read_csv('data/all_pollutants_daily_clean_2023-2025.csv')
combined['Date Local'] = pd.to_datetime(combined['Date Local'])

# Game dates array
game_dates = pd.to_datetime([
    "2023-09-11", "2023-09-24", "2023-10-01", "2023-10-15", "2023-11-06", 
    "2023-11-24", "2023-12-03", "2023-12-10", "2023-12-24", "2024-09-19", 
    "2024-09-29", "2024-10-14", "2024-10-31", "2024-11-17", "2024-12-01", 
    "2024-12-22", "2025-01-05", "2025-09-07", "2025-09-14", "2025-10-05", 
    "2025-10-19", "2025-11-09", "2025-11-30", "2025-12-07", "2025-12-28", 
    "2023-09-10", "2023-10-02", "2023-10-22", "2023-10-29", "2023-11-26", 
    "2023-12-11", "2023-12-31", "2024-01-07", "2024-09-08", "2024-09-26", 
    "2024-10-13", "2024-10-20", "2024-11-03", "2024-11-24", "2024-12-08", 
    "2024-12-15", "2024-12-28", "2025-09-21", "2025-09-28", "2025-10-09", 
    "2025-10-26", "2025-11-02", "2025-11-16", "2025-12-14", "2025-12-21" 
])

nfl_df = combined[
    combined['Date Local'].isin(game_dates)
].copy()

nfl_df['overall_aqi'] = nfl_df.groupby(
    ['Local Site Name', 'Date Local']
)['daily_max_aqi'].transform('max')

def get_main_pollutant(group):
    max_aqi = group['daily_max_aqi'].max()
    top = group[group['daily_max_aqi'] == max_aqi]['pollutant'].tolist()
    return ' / '.join(sorted(top)) if len(top) > 1 else top[0]

main_poll = nfl_df.groupby(
    ['Local Site Name', 'Date Local']
).apply(get_main_pollutant).reset_index()
main_poll.columns = ['Local Site Name', 'Date Local', 'main_pollutant']

nfl_df = nfl_df.drop(columns='main_pollutant', errors='ignore').merge(
    main_poll, on=['Local Site Name', 'Date Local']
)

nfl_df.to_csv('NFL_gameday_airquality.csv', index=False)